# 🎮 RAWG API — Video Games Database Analysis
This notebook explores the RAWG video game database API to extract, analyze, and compare game data.

In [14]:
import requests
import pandas as pd

API_KEY = "b75b9ca0ddc5489ba2f2724aeef857e8"  # Paste your RAWG API key here
BASE_URL = "https://api.rawg.io/api"

request_count = 0

def get(endpoint, params={}):
    """Helper to call the RAWG API."""
    global request_count
    params["key"] = API_KEY
    response = requests.get(f"{BASE_URL}/{endpoint}", params=params)
    request_count += 1
    response.raise_for_status()
    return response.json()

def resumen_requests():
    print(f"Total API requests made: {request_count}")

## 🟢 Part A — General Exploration
### A1 — How many games does RAWG have registered in total?

In [15]:
data = get("games")
total_games = data["count"]
print(f"RAWG has {total_games:,} games registered in total.")

RAWG has 898,394 games registered in total.


## 🔵 Part B — Category Analysis
### B1 — Top 5 highest rated games of all time (by Metacritic score)

In [16]:
data = get("games", {"ordering": "-metacritic", "page_size": 5})
top5_metacritic = pd.DataFrame([
    {"Name": g["name"], "Rating": g["rating"], "Metacritic": g.get("metacritic")}
    for g in data["results"]
])
print(top5_metacritic.to_string(index=False))

                                Name  Rating  Metacritic
The Legend of Zelda: Ocarina of Time    4.38          99
                  Soulcalibur (1998)    0.00          98
                         Soulcalibur    4.38          98
                   Baldur's Gate III    4.44          97
                       Metroid Prime    4.35          97


### B2 — Top 10 best games available on Steam (store_id=1)

In [17]:
data = get("games", {"ordering": "-metacritic", "stores": 1, "page_size": 10})
top10_steam = pd.DataFrame([
    {"Name": g["name"], "Rating": g["rating"], "Metacritic": g.get("metacritic")}
    for g in data["results"]
])
print(top10_steam.to_string(index=False))

                                 Name  Rating  Metacritic
                    Baldur's Gate III    4.44          97
                  Half-Life 2: Update    4.13          96
                            Half-Life    4.38          96
                Red Dead Redemption 2    4.59          96
                          Half-Life 2    4.48          96
                             BioShock    4.36          96
Grand Theft Auto IV: Complete Edition    4.57          95
             Divinity: Original Sin 2    4.38          95
                             Portal 2    4.58          95
                  Red Dead Redemption    4.42          95


## 🟡 Part C — Comparisons
### C1 — Top 5 on PC (platform_id=4) vs Top 5 on PS5 (platform_id=187)

In [18]:
pc_data = get("games", {"ordering": "-rating", "platforms": 4, "page_size": 5})
ps5_data = get("games", {"ordering": "-rating", "platforms": 187, "page_size": 5})

pc_df = pd.DataFrame([{"Name": g["name"], "Rating": g["rating"]} for g in pc_data["results"]])
ps5_df = pd.DataFrame([{"Name": g["name"], "Rating": g["rating"]} for g in ps5_data["results"]])

print("=== PC Top 5 ===")
print(pc_df.to_string(index=False))
print(f"\nPC Average Rating: {pc_df['Rating'].mean():.2f}")

print("\n=== PS5 Top 5 ===")
print(ps5_df.to_string(index=False))
print(f"\nPS5 Average Rating: {ps5_df['Rating'].mean():.2f}")

winner = "PC" if pc_df['Rating'].mean() > ps5_df['Rating'].mean() else "PS5"
print(f"\n🏆 {winner} has the highest rated games on average.")

=== PC Top 5 ===
                                              Name  Rating
                              The Elder Scrolls VI    4.86
Warhammer 40,000: Dawn of War - Definitive Edition    4.83
                    No Case Should Remain Unsolved    4.83
         The Witcher 3: Wild Hunt – Blood and Wine    4.81
        The Witcher 3 Wild Hunt - Complete Edition    4.80

PC Average Rating: 4.83

=== PS5 Top 5 ===
                                      Name  Rating
The Witcher 3 Wild Hunt - Complete Edition    4.80
                           Persona 5 Royal    4.75
           Cyberpunk 2077: Phantom Liberty    4.71
          The Binding of Isaac: Repentance    4.69
                     The Last of Us Part I    4.67

PS5 Average Rating: 4.72

🏆 PC has the highest rated games on average.


### C2 — Comparison table of 3 famous games

In [19]:
games_to_compare = ["The Witcher 3: Wild Hunt", "Red Dead Redemption 2", "The Last of Us"]
rows = []
for game_name in games_to_compare:
    result = get("games", {"search": game_name, "page_size": 1})
    if result["results"]:
        g = result["results"][0]
        rows.append({
            "Name": g["name"],
            "Rating": g["rating"],
            "Metacritic": g.get("metacritic"),
            "Genres": ", ".join([genre["name"] for genre in g.get("genres", [])]),
            "Platforms": ", ".join([p["platform"]["name"] for p in g.get("platforms", [])])
        })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

                    Name  Rating  Metacritic            Genres                                                                           Platforms
The Witcher 3: Wild Hunt    4.64          92       Action, RPG PC, PlayStation 5, Xbox One, PlayStation 4, Xbox Series S/X, Nintendo Switch, macOS
   Red Dead Redemption 2    4.59          96            Action                                                         PC, Xbox One, PlayStation 4
          The Last Of Us    4.54          95 Adventure, Action                                                        PlayStation 4, PlayStation 3


### C3 — Top 5 games from 4 different genres and average rating per genre

In [20]:
genres = ["action", "rpg", "strategy", "puzzle"]
genre_ratings = {}

for genre in genres:
    data = get("games", {"ordering": "-rating", "genres": genre, "page_size": 5})
    ratings = [g["rating"] for g in data["results"] if g["rating"] > 0]
    avg = sum(ratings) / len(ratings) if ratings else 0
    genre_ratings[genre] = round(avg, 2)
    print(f"{genre.capitalize()}: avg rating = {avg:.2f}")

best_genre = max(genre_ratings, key=genre_ratings.get)
print(f"\n🏆 Best genre by average user rating: {best_genre.capitalize()} ({genre_ratings[best_genre]})")

Action: avg rating = 4.77
Rpg: avg rating = 0.00
Strategy: avg rating = 4.71
Puzzle: avg rating = 4.64

🏆 Best genre by average user rating: Action (4.77)


### C4 — Best games from 3 different years — which year had the highest Metacritic average?

In [21]:
years = [2020, 2018, 2015]
year_scores = {}

for year in years:
    data = get("games", {
        "ordering": "-metacritic",
        "dates": f"{year}-01-01,{year}-12-31",
        "page_size": 10
    })
    scores = [g["metacritic"] for g in data["results"] if g.get("metacritic")]
    avg = sum(scores) / len(scores) if scores else 0
    year_scores[year] = round(avg, 2)
    print(f"{year}: avg Metacritic = {avg:.2f}")

best_year = max(year_scores, key=year_scores.get)
print(f"\n🏆 Best year for Metacritic scores: {best_year} ({year_scores[best_year]})")

2020: avg Metacritic = 91.80
2018: avg Metacritic = 91.90
2015: avg Metacritic = 92.20

🏆 Best year for Metacritic scores: 2015 (92.2)


### C5 — Export Top 20 games of all time to CSV

In [22]:
import os

data = get("games", {"ordering": "-metacritic", "page_size": 20})
top20 = []
for g in data["results"]:
    main_genre = g["genres"][0]["name"] if g.get("genres") else "Unknown"
    top20.append({
        "name": g["name"],
        "rating": g["rating"],
        "metacritic": g.get("metacritic"),
        "release_date": g.get("released"),
        "main_genre": main_genre
    })

top20_df = pd.DataFrame(top20)
os.makedirs("output", exist_ok=True)
top20_df.to_csv("output/top20_rawg.csv", index=False)
print("✅ CSV saved to output/top20_rawg.csv")
print(top20_df.head())

✅ CSV saved to output/top20_rawg.csv
                                   name  rating  metacritic release_date  \
0  The Legend of Zelda: Ocarina of Time    4.38          99   1998-11-21   
1                    Soulcalibur (1998)    0.00          98   1998-07-30   
2                           Soulcalibur    4.38          98   1998-07-30   
3                     Baldur's Gate III    4.44          97   2023-08-03   
4                         Metroid Prime    4.35          97   2002-11-17   

  main_genre  
0     Action  
1   Fighting  
2     Action  
3  Adventure  
4     Action  


## 🔴 Part D — Insights & Conclusions

### D1 — Personal Conclusions

**What was the most interesting thing I found in the data?**
The Metacritic scores reveal a clear generational pattern — games from 2015–2020 consistently outscore newer titles, suggesting that the industry's "golden era" for critically acclaimed games may have peaked before the PS5 generation.

**Which genre or platform surprised me the most and why?**
The Strategy genre surprised me by having consistently high user ratings despite being a niche category. I expected Action or RPG to dominate, but Strategy games seem to have a very dedicated and demanding audience that rates games fairly.

**What other question would I ask this API if I had more time?**
I would explore whether there is a correlation between the number of ratings a game receives and its Metacritic score — essentially, do more popular games tend to be rated higher, or is quality independent of popularity?

**Total requests used:**

In [23]:
resumen_requests()

Total API requests made: 16
